# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DEEPIKA-Inag/Flyrank-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*
### My approach

I build the feature vector straight from `data/raw/content_refresh_anonymized.csv` (the code
below reads it from the repo if it's cloned locally, otherwise falls back to the raw GitHub URL
so the notebook still runs standalone in Colab).

**Engineering choices:**
- **Heavy-tailed counts** (`impressions_90d`, `clicks_90d`, `sessions_90d`, `ai_sessions_90d`) get
  a `log1p` transform instead of being used raw — a handful of very-high-traffic pages would
  otherwise dominate a linear model.
- **Structural missingness gets its own flag, not a silent zero.** The dictionary warns that
  `feedly article` rows have *no* keyword data and ~28% of `keyword article` rows have no
  `word_count` — a blind `fillna(0)` would quietly teach the model "0 search volume" instead of
  "we never measured this." So I add `has_keyword_data` / `has_word_count` **before** filling the
  numeric columns with 0, then fill.
- **Categoricals** (`content_type`, `main_intent`, `competition_level`, the tier columns) are
  one-hot encoded; blanks become `"unknown"` rather than being dropped, since "unknown" is itself
  informative (e.g. `main_intent` blank vs. a real intent).
- **Rates** (`ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`) are used as-is — they're
  already normalized and, per the dictionary, `scroll_rate`/`ai_traffic_pct` can legitimately
  exceed 100 because numerator and denominator come from different measurement systems, so I
  don't clip them.
- **Nothing that touches the label window makes it in here.** `trend_direction`, `trend_pct`, and
  the raw `*_last_30d` / `*_prev_30d` columns are left out of the feature set entirely — Section 3
  shows why, Section 4 lists every exclusion with a reason.


In [1]:
import os
import numpy as np
import pandas as pd

RAW_LOCAL = "data/raw/content_refresh_anonymized.csv"
RAW_URL = (
    "https://raw.githubusercontent.com/DEEPIKA-Inag/Flyrank-internship/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(RAW_LOCAL if os.path.exists(RAW_LOCAL) else RAW_URL)
print(f"raw shape: {df.shape}")

# --- columns that get a plain numeric fill (0 = "no data") -----------------
NUMERIC_FILL_ZERO = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "age_tier_order", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct", "trend_pct",
]
CATEGORICAL_COLS = [
    "competition_level", "content_type", "main_intent", "provider_used", "model_used",
    "age_tier", "freshness_tier", "word_count_tier", "char_count_tier",
    "impression_tier", "position_tier", "trend_direction",
]

feat = df.copy()
for c in NUMERIC_FILL_ZERO:
    feat[c] = pd.to_numeric(feat[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
for c in CATEGORICAL_COLS:
    feat[c] = feat[c].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})

# has_-flags BEFORE the fillna(0), so structural missingness stays visible to the model
feat["has_keyword_data"] = df["search_volume"].notna().astype(int)
feat["has_word_count"] = df["word_count"].notna().astype(int)

for c in NUMERIC_FILL_ZERO:
    feat[c] = feat[c].fillna(0)

# engineered numeric features
feat["log_impressions_90d"] = np.log1p(feat["impressions_90d"])
feat["log_clicks_90d"] = np.log1p(feat["clicks_90d"])
feat["log_sessions_90d"] = np.log1p(feat["sessions_90d"])
feat["log_ai_sessions_90d"] = np.log1p(feat["ai_sessions_90d"])
feat["has_clicks"] = (feat["clicks_90d"] > 0).astype(int)
feat["has_ai_sessions"] = (feat["ai_sessions_90d"] > 0).astype(int)
feat["measurable_opportunity"] = (
    (feat["impressions_90d"] >= 100) & (feat["sessions_90d"] > 0)
).astype(int)

# the label — built here only so Section 3 can test against it; NEVER a feature
feat["is_declining_label"] = (feat["trend_direction"].str.lower() == "down").astype(int)

# the feature set this notebook stands behind (Section 4 explains every exclusion)
SAFE_NUMERIC = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "has_keyword_data", "has_word_count", "has_clicks", "has_ai_sessions", "measurable_opportunity",
]
SAFE_CATEGORICAL = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

print(f"feature vector: {len(SAFE_NUMERIC)} numeric + {len(SAFE_CATEGORICAL)} categorical columns")
print(f"base rate of is_declining_label: {feat['is_declining_label'].mean():.3f}")
feat[["content_id", "client_id"] + SAFE_NUMERIC + SAFE_CATEGORICAL + ["is_declining_label"]].head(3)


raw shape: (30000, 44)
feature vector: 23 numeric + 8 categorical columns
base rate of is_declining_label: 0.542


,content_id,client_id,search_volume,competition,cpc,word_count,char_count,log_impressions_90d,log_clicks_90d,log_sessions_90d,...,measurable_opportunity,competition_level,content_type,main_intent,age_tier,freshness_tier,word_count_tier,impression_tier,position_tier,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,2.05,3221.0,20457.0,8.243808,3.401197,2.890372,...,1,HIGH,keyword article,transactional,181-365,0-30,2000-3500,good,striking,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,0.05,2481.0,15562.0,9.636980,2.079442,2.302585,...,1,LOW,keyword article,informational,365+,0-30,2000-3500,good,page_3_5,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,0.00,3515.0,23643.0,9.440023,2.484907,2.484907,...,1,LOW,keyword article,informational,91-180,0-30,3500+,good,page_3_5,1


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*
### Feature notes

Every column below is one this notebook actually feeds to a model (Section 1). "Available
before?" asks: could I have known this value at the moment I'd need to make the prediction, i.e.
does it live strictly inside the 90-day feature window and outside the last/prev-30d label
window? Section 2's code cell backs the missing-value claims with real counts.

| Feature | Meaning | Missing values | Available before label window? |
|---|---|---|---|
| `search_volume` | Search-volume estimate for the target keyword | Blank for `feedly article` (100%) and a few `keyword article` rows (1.4%) → filled 0, flagged by `has_keyword_data` | Yes — keyword metadata, set at authoring time |
| `competition` | Keyword competition score 0–1 | Same pattern as `search_volume` → filled 0 | Yes |
| `cpc` | Cost-per-click estimate | Same pattern → filled 0 | Yes |
| `word_count` | Article length | Blank for 28% of `keyword article` rows (not measured) → filled 0, flagged by `has_word_count` | Yes |
| `char_count` | Article length in characters | Blank alongside `word_count` → filled 0 | Yes |
| `log_impressions_90d` | log1p of 90-day GSC impressions | None (0 impressions is a valid, if excluded, value) | Yes — trailing 90d window, ends before the last-30d label window starts |
| `log_clicks_90d` | log1p of 90-day GSC clicks | None | Yes, same window |
| `log_sessions_90d` | log1p of 90-day GA4 sessions | None | Yes, same window |
| `log_ai_sessions_90d` | log1p of 90-day AI-referred sessions | None | Yes, same window |
| `days_with_impressions` | Days (0–90) with ≥1 impression | None | Yes |
| `days_with_sessions` | Days (0–90) with ≥1 session | None | Yes |
| `content_age_days` | Days since the page was created | None (every row ≥ 90) | Yes |
| `days_since_last_update` | Days since last content update | None | Yes |
| `ctr` | `clicks_90d / impressions_90d × 100` | None | Yes — computed over the full 90d window, which overlaps but is not defined by the last/prev 30d split |
| `avg_position` | Mean GSC position; **0 = "no data"**, not rank 0 | None (0 is the sentinel, left as-is + captured by `impression_tier`/`position_tier`) | Yes |
| `engagement_rate` | `engaged_sessions_90d / sessions_90d × 100` | None | Yes |
| `scroll_rate` | `scroll_events_90d / pageviews_90d × 100`, can exceed 100 | Blank when `pageviews_90d = 0` → filled 0 | Yes |
| `ai_traffic_pct` | `ai_sessions_90d / sessions_90d × 100`, can exceed 100 | None | Yes |
| `has_keyword_data` | 1 if `search_volume` was measured | — (engineered flag) | Yes |
| `has_word_count` | 1 if `word_count` was measured | — (engineered flag) | Yes |
| `has_clicks` | 1 if `clicks_90d > 0` | — | Yes |
| `has_ai_sessions` | 1 if `ai_sessions_90d > 0` | — | Yes |
| `measurable_opportunity` | 1 if `impressions_90d ≥ 100` and `sessions_90d > 0` | — | Yes |
| `competition_level` | LOW/MEDIUM/HIGH bucket of `competition` | Blank → `"unknown"` | Yes |
| `content_type` | keyword / feedly / comparison article | None | Yes |
| `main_intent` | informational / transactional / commercial / navigational | Blank → `"unknown"` | Yes |
| `age_tier` | bucketed `content_age_days` | None | Yes |
| `freshness_tier` | bucketed `days_since_last_update` | None | Yes |
| `word_count_tier` | bucketed `word_count` | Blank → `"unknown"` | Yes |
| `impression_tier` | bucketed `impressions_90d` (incl. `no_data`) | None | Yes |
| `position_tier` | bucketed `avg_position` (incl. `no_data`) | None | Yes |

`content_id` / `client_id` are carried along for joins and the grouped split only — never passed
to the model as features (Section 4).


In [2]:
# Back the "missing values" claims above with real counts, and confirm the
# missingness is systematic (tracks content_type), not random -- exactly the
# trap the data dictionary warns about.
missing_counts = df[[
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "scroll_rate", "trend_pct",
]].isna().sum().sort_values(ascending=False)
print("missing value counts:")
print(missing_counts.to_string())

print("\nsearch_volume missing rate by content_type:")
print(df.groupby("content_type")["search_volume"].apply(lambda s: s.isna().mean()).round(3).to_string())

print("\nword_count missing rate by content_type:")
print(df.groupby("content_type")["word_count"].apply(lambda s: s.isna().mean()).round(3).to_string())

# sanity check: avg_position's 0 sentinel count matches the dictionary (1,205 rows)
print(f"\navg_position == 0 (no position data): {(df['avg_position'] == 0).sum()} rows")

# confirm every SAFE_NUMERIC / SAFE_CATEGORICAL column ends up with zero nulls
# after the Section 1 fills -- the "available before" table is only honest if
# nothing downstream still has holes in it.
final_nulls = feat[SAFE_NUMERIC + SAFE_CATEGORICAL].isna().sum().sum()
print(f"\nnull cells remaining in the model-ready feature matrix: {final_nulls}")
assert final_nulls == 0, "feature table still has nulls -- the notes above are wrong"


missing value counts:
word_count       7699
char_count       7699
trend_pct        3388
cpc              2468
competition      2468
search_volume    2468
scroll_rate       125

search_volume missing rate by content_type:
content_type
comparison article    0.000
feedly article        1.000
keyword article       0.014

word_count missing rate by content_type:
content_type
comparison article    0.000
feedly article        0.000
keyword article       0.283

avg_position == 0 (no position data): 1205 rows

null cells remaining in the model-ready feature matrix: 0


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*
### What I found

I attacked the feature set the way the checklist says to: **add a suspect back in and watch the
score jump toward 1.0.** Everything is trained as `is_declining_label` (54.2% positive — printed
below, that's the number every AUC has to beat) with logistic regression, 5-fold **grouped**
cross-validation on `client_id` (never a random split — 32 clients repeat many rows each, so a
random split lets the model memorize a client instead of learning the pattern).

| Feature set | Grouped-CV AUC | What it means |
|---|---|---|
| Honest set (Section 1's 31 columns) | **0.660** | Real, modest signal — nowhere near perfect |
| + `trend_pct` | **0.999** | `trend_pct` directly parameterizes `trend_direction`, which *is* the label. Near-1.0 is the textbook confession. |
| + `impressions_last_30d` / `impressions_prev_30d` | **0.907** | These two columns are the raw inputs `trend_pct` is computed from — same leak, one step upstream. |

**Timeline:** every honest feature is a 90-day trailing aggregate (or something derived from
authoring metadata) that closes *before* the comparison the label is built on. `trend_pct` and
the `*_last_30d` / `*_prev_30d` columns exist specifically to compare "the most recent 30 days"
against "the 30 days before that" — that comparison **is** the label. There's no legal way to
split those columns into a safe sub-window the way the leakage skill suggests for overlapping
aggregates, because their entire purpose is describing the label window itself. They're out
entirely (Section 4).

**One more thing worth reporting:** grouped-CV AUC on the honest set is 0.660; an ungrouped
random-split AUC on the *same* features comes out higher, at 0.718. A 0.058 gap between random
and grouped splits on identical features means the model is partly memorizing per-client
baseline behavior rather than purely reading page-level signal — small, but real, and it's why
the grouped split (not the random one) is the number I'd report as "the" score.


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print(f"base rate (is_declining_label): {feat['is_declining_label'].mean():.3f}")


def grouped_cv_auc(numeric_cols, categorical_cols, label):
    X = feat[numeric_cols + categorical_cols]
    y = feat["is_declining_label"]
    groups = feat["client_id"]
    pipe = Pipeline([
        ("pre", ColumnTransformer([
            ("num", StandardScaler(), numeric_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ])),
        ("clf", LogisticRegression(max_iter=1000)),
    ])
    aucs = []
    for tr_idx, te_idx in GroupKFold(n_splits=5).split(X, y, groups):
        pipe.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        p = pipe.predict_proba(X.iloc[te_idx])[:, 1]
        aucs.append(roc_auc_score(y.iloc[te_idx], p))
    print(f"{label}: grouped-CV AUC = {np.mean(aucs):.3f} (+/- {np.std(aucs):.3f})")
    return float(np.mean(aucs))


print("\n--- the attack: add suspects back in, one group at a time ---")
honest_auc = grouped_cv_auc(SAFE_NUMERIC, SAFE_CATEGORICAL, "honest set (no leakage)")
trendpct_auc = grouped_cv_auc(SAFE_NUMERIC + ["trend_pct"], SAFE_CATEGORICAL, "+ trend_pct")
window_auc = grouped_cv_auc(
    SAFE_NUMERIC + ["impressions_last_30d", "impressions_prev_30d"],
    SAFE_CATEGORICAL,
    "+ last/prev 30d impressions",
)

print("\n--- random split vs grouped split, same honest features ---")
X = feat[SAFE_NUMERIC + SAFE_CATEGORICAL]
y = feat["is_declining_label"]
pipe = Pipeline([
    ("pre", ColumnTransformer([
        ("num", StandardScaler(), SAFE_NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore"), SAFE_CATEGORICAL),
    ])),
    ("clf", LogisticRegression(max_iter=1000)),
])
random_aucs = []
for tr_idx, te_idx in StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X, y):
    pipe.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    p = pipe.predict_proba(X.iloc[te_idx])[:, 1]
    random_aucs.append(roc_auc_score(y.iloc[te_idx], p))
random_auc = float(np.mean(random_aucs))
print(f"random-split AUC: {random_auc:.3f}  vs  grouped-split AUC: {honest_auc:.3f}"
      f"  (gap = {random_auc - honest_auc:.3f})")


base rate (is_declining_label): 0.542

--- the attack: add suspects back in, one group at a time ---
honest set (no leakage): grouped-CV AUC = 0.660 (+/- 0.039)
+ trend_pct: grouped-CV AUC = 0.999 (+/- 0.001)
+ last/prev 30d impressions: grouped-CV AUC = 0.907 (+/- 0.042)

--- random split vs grouped split, same honest features ---
random-split AUC: 0.718  vs  grouped-split AUC: 0.660  (gap = 0.058)


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*
### Exclusions

The code cell below is the actual test: it lists every raw column, states whether it feeds the
model directly, feeds it through a transform, or was refused — and asserts the three groups add
up to all 44 raw columns, so nothing was quietly forgotten.

Reasons, in short:

- **IDs — grouping only, never features:** `content_id`, `client_id`. Pseudonyms carry no
  content signal; `client_id` is used exclusively to build the `GroupKFold` split.
- **Label-derived — this *is* the leak (Section 3 proved it):** `trend_direction` (the label
  itself), `trend_pct`, and the six `*_last_30d` / `*_prev_30d` columns that `trend_pct` is
  computed from.
- **Not a signal about performance — an authoring/ops artifact:** `provider_used`, `model_used`.
  The data dictionary calls these out directly as "not a model feature" — which LLM wrote the
  article doesn't causally drive whether search demand for it later declines.
- **Redundant with a column already in the set:** `age_tier_order` (same information as
  `age_tier`, just re-encoded as an integer); `char_count_tier` (96%+ of `2000-3500` word-count
  rows fall in the same `15000-25000` char bucket — keeping both tiers just double-counts length);
  `users_90d` (corr. 0.998 with `sessions_90d`, already used via `log_sessions_90d`);
  `pageviews_90d` and `engaged_sessions_90d`/`scroll_events_90d` (already expressed as
  `engagement_rate` / `scroll_rate`, which are the versions I kept).


In [4]:
# every raw column, sorted into exactly one bucket -- and the assertion below
# is the actual test: if anything is missing or double-counted, this fails.
used_directly = {
    "search_volume", "competition", "competition_level", "cpc", "content_type", "main_intent",
    "word_count", "char_count", "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "age_tier", "freshness_tier", "word_count_tier",
    "impression_tier", "position_tier",
}
used_via_transform = {"impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"}

excluded_reasons = {
    "content_id": "pseudonym -- grouping/joins only, no signal",
    "client_id": "pseudonym -- used only to build the GroupKFold split, never a feature",
    "trend_direction": "IS the label (is_declining_label = trend_direction == 'down')",
    "trend_pct": "computes trend_direction -- Section 3: AUC 0.660 -> 0.999 when included",
    "impressions_last_30d": "half of the trend_pct formula",
    "clicks_last_30d": "same last-30d window the label is built from",
    "sessions_last_30d": "same last-30d window the label is built from",
    "impressions_prev_30d": "other half of the trend_pct formula",
    "clicks_prev_30d": "same prev-30d window the label is built from",
    "sessions_prev_30d": "same prev-30d window the label is built from",
    "provider_used": "which LLM wrote the article -- an authoring artifact, not a cause of decline; dictionary flags it 'not a model feature'",
    "model_used": "same reasoning as provider_used",
    "age_tier_order": "integer re-encoding of age_tier -- including both double-counts one signal",
    "char_count_tier": "96%+ overlaps word_count_tier bucket-for-bucket -- redundant, kept word_count_tier",
    "pageviews_90d": "already expressed as scroll_rate's denominator; corr 0.97 with sessions_90d",
    "users_90d": f"near-duplicate of sessions_90d (corr {df['users_90d'].corr(df['sessions_90d']):.3f}), already used via log_sessions_90d",
    "engaged_sessions_90d": "raw numerator already expressed as engagement_rate",
    "scroll_events_90d": "raw numerator already expressed as scroll_rate",
}

all_raw = set(df.columns)
accounted = used_directly | used_via_transform | set(excluded_reasons)
assert accounted == all_raw, (
    f"missing: {all_raw - accounted}  extra: {accounted - all_raw}"
)

print(f"{len(df.columns)} raw columns total")
print(f"{len(used_directly)} used directly + {len(used_via_transform)} used via transform "
      f"= {len(used_directly) + len(used_via_transform)} feeding the model")
print(f"{len(excluded_reasons)} excluded:\n")
for col, reason in excluded_reasons.items():
    print(f"  {col}: {reason}")


44 raw columns total
22 used directly + 4 used via transform = 26 feeding the model
18 excluded:

  content_id: pseudonym -- grouping/joins only, no signal
  client_id: pseudonym -- used only to build the GroupKFold split, never a feature
  trend_direction: IS the label (is_declining_label = trend_direction == 'down')
  trend_pct: computes trend_direction -- Section 3: AUC 0.660 -> 0.999 when included
  impressions_last_30d: half of the trend_pct formula
  clicks_last_30d: same last-30d window the label is built from
  sessions_last_30d: same last-30d window the label is built from
  impressions_prev_30d: other half of the trend_pct formula
  clicks_prev_30d: same prev-30d window the label is built from
  sessions_prev_30d: same prev-30d window the label is built from
  provider_used: which LLM wrote the article -- an authoring artifact, not a cause of decline; dictionary flags it 'not a model feature'
  model_used: same reasoning as provider_used
  age_tier_order: integer re-encoding 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.